<a href="https://colab.research.google.com/github/lubbad-aljazeera/paid-activity-data-pipeline/blob/main/X_Data_Pipeline_Extend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from google.cloud import storage, bigquery
from google.colab import auth
import re
import io
import numpy as np
import csv
import traceback # Import the traceback module
from datetime import datetime # Import datetime for backup table naming

auth.authenticate_user()
print("✅ Google Cloud authenticated successfully.")

bucket_name = "aj360_data"
dataset_id = "aj360"
table_id = "x_ads_performance"
project_id = "ajgc-dig-jwc-prod-jawacloud-01"
dataset_ref = bigquery.DatasetReference(project_id, dataset_id)
table_ref = dataset_ref.table(table_id)
clean_file = "x_ads_clean.csv"

prefix = "X_ADS/x_ads_performance_5-7Mar2026.csv"
print(f"Listing CSV files in bucket: {bucket_name} with prefix: {prefix}")
storage_client = storage.Client(project=project_id)
bucket = storage_client.bucket(bucket_name)
blobs = list(bucket.list_blobs(prefix=prefix))
csv_files = [blob.name for blob in blobs if blob.name.endswith('.csv')]

def extract_hashtags(text):
    # Ensure text is treated as string, handle potential non-string types or NaN
    text = str(text) if pd.notna(text) else ""
    hashtags = re.findall(r'#(\w+)', text)
    return ','.join(hashtags) if hashtags else None # Join with comma, no space to be safe for CSV

# Function to clean column names for BigQuery compatibility
def clean_column_name_for_bq(col_name):
    # Replace non-alphanumeric (except underscore) with underscore, then lowercase
    cleaned = re.sub(r'[^a-zA-Z0-9_]', '_', str(col_name))
    # Remove leading/trailing underscores and multiple consecutive underscores
    cleaned = re.sub(r'_{2,}', '_', cleaned).strip('_')
    return cleaned.lower()

if not csv_files:
    print("❌ No CSV files found.")
else:
    print(f"✅ Found {len(csv_files)} CSV files to process.")

    bq_client = bigquery.Client(project=project_id)

    # --- Dataset Check/Creation ---
    try:
        bq_client.get_dataset(dataset_ref)
        print(f"✅ Dataset {dataset_id} exists.")
    except:
        dataset = bigquery.Dataset(dataset_ref)
        dataset.location = "US"
        dataset = bq_client.create_dataset(dataset)
        print(f"✅ Created dataset {dataset_id}.")

    print("Loading lookup table from BigQuery...")
    try:
        lookup_df = bq_client.query("""
            SELECT term, show_title_ar, show_title_en
            FROM `ajgc-dig-jwc-prod-jawacloud-01.aj360.paid_ads_shows_lookup`
        """).to_dataframe()
        term_lookup = dict(zip(lookup_df['term'], lookup_df['show_title_en']))
        term_lookup_ar = dict(zip(lookup_df['term'], lookup_df['show_title_ar']))

        # NEW: Create a direct mapping from Arabic show titles to English show titles for hashtags
        arabic_title_to_english_title_map = {}
        for _, row in lookup_df.iterrows():
            # Clean Arabic title for consistent lookup: remove spaces and underscores
            cleaned_arabic_title = str(row['show_title_ar']).strip().replace(' ', '').replace('_', '')
            english_title = str(row['show_title_en']).strip()
            arabic_title_to_english_title_map[cleaned_arabic_title] = english_title

        print(f"✅ Loaded lookup table with {len(term_lookup)} terms.")
    except Exception as e:
        print(f"❌ Failed to load lookup table: {e}")
        csv_files = [] # Prevent further processing if lookup fails

    bq_target_schema = None # This will hold the BigQuery table's final schema

    # --- Schema Handling: Prioritize existing BQ table schema ---
    try:
        table = bq_client.get_table(table_ref)
        bq_target_schema = table.schema
        print(f"✅ Retrieved existing BigQuery table schema for {table_id}.")
    except Exception as e:
        print(f"⚠️ BigQuery table {table_id} does not exist or error retrieving schema: {e}.")
        print("   Proceeding to infer schema from CSV files to create the table.")

        # If table does not exist, infer schema from CSVs
        unified_schema = {}
        print("\n--- Schema Inference from CSVs ---")
        for file in csv_files[:5]: # Infer from a sample of files
            try:
                print(f"Inferring schema from: {file}")
                blob = bucket.blob(file)
                blob.download_to_filename("/tmp/schema.csv")
                df_temp = pd.read_csv("/tmp/schema.csv", encoding='utf-8', on_bad_lines='skip')

                if df_temp.empty:
                    continue

                df_temp.columns = [clean_column_name_for_bq(col) for col in df_temp.columns]

                # Ensure 'show_title' is included in schema inference if it's a new column
                if ('ad_group_name' in df_temp.columns or 'ad_name' in df_temp.columns or 'tweet_text' in df_temp.columns) and 'show_title' not in df_temp.columns:
                    df_temp['show_title'] = None

                for col, dtype in df_temp.dtypes.items():
                    if col not in unified_schema:
                        if dtype == 'object':
                            bq_type = 'STRING'
                        elif 'int' in str(dtype):
                            bq_type = 'INTEGER'
                        elif 'float' in str(dtype):
                            bq_type = 'FLOAT'
                        elif 'bool' in str(dtype):
                            bq_type = 'BOOLEAN'
                        elif 'datetime' in str(dtype):
                            bq_type = 'TIMESTAMP'
                        else:
                            bq_type = 'STRING'
                        unified_schema[col] = bq_type
            except Exception as infer_e:
                print(f"❌ Failed schema inference on {file}: {infer_e}")

        if not unified_schema:
            print("❌ Schema inference failed. Exiting.")
            exit()

        # Convert inferred schema dictionary to BigQuery SchemaField list
        bq_target_schema = [
            bigquery.SchemaField(k, v) # Names already cleaned by clean_column_name_for_bq
            for k, v in unified_schema.items()
        ]

        # Create the BigQuery table with the inferred schema
        try:
            bq_client.create_table(bigquery.Table(table_ref, schema=bq_target_schema))
            print(f"✅ Created BigQuery table {table_id} with inferred schema.")
        except Exception as create_e:
            print(f"❌ Failed to create BigQuery table {table_id}: {create_e}")
            print("   Attempting to retrieve existing table schema, assuming it was just created concurrently.")
            try: # Try to fetch its schema, in case it was created by a concurrent run
                table = bq_client.get_table(table_ref)
                bq_target_schema = table.schema
                print(f"✅ Table already existed, retrieved its schema.")
            except Exception as final_e:
                print(f"❌ Critical error: Could not get or create BigQuery table schema: {final_e}. Exiting.")
                exit()

    # 'schema' variable will now consistently hold the definitive BigQuery table schema.
    schema = bq_target_schema

    if not schema: # Should not happen if previous steps are successful
        print("❌ Final BigQuery schema not determined. Exiting.")
        exit()

    print("\n✅ Final Schema for BigQuery operations:")
    for s in schema:
        print(f"  {s.name}: {s.field_type}")

    if csv_files and schema: # Only proceed with data processing if files and schema are available
      # Generates a string like '2026_02_26' (Recommended for better sorting)
      current_date_string = datetime.now().strftime('%Y_%m_%d')
      backup_table_name = f"{table_id}_backup_{current_date_string}"
      backup_table_ref = dataset_ref.table(backup_table_name)
      backup_query = f"""
      CREATE TABLE IF NOT EXISTS `{project_id}.{dataset_id}.{backup_table_name}1` AS
      SELECT * FROM `{project_id}.{dataset_id}.{table_id}`;
      """
      print(f"\n--- Creating BigQuery Backup Table: {backup_table_name} ---")
      try:
          bq_client.query(backup_query).result()
          print(f"✅ Backup table '{backup_table_name}' created successfully (if it didn't exist) or already exists.")
      except Exception as e:
          print(f"❌ Failed to create backup table '{backup_table_name}': {e}")
          # Decide if processing should halt on backup failure; for now, print error and continue
      # --- End Backup Mechanism ---'

    print("\n--- Starting Data Processing and Loading Pass ---")
    for i, source_file in enumerate(csv_files):
        print(f"\nProcessing: {source_file}")
        try:
            blob = bucket.blob(source_file)
            blob.download_to_filename("/tmp/source.csv")
            df = pd.read_csv("/tmp/source.csv", encoding='utf-8', on_bad_lines='skip')

            if df.empty:
                print("⚠️ Empty file. Skipping.")
                continue

            # Clean DataFrame columns to match BigQuery compatibility standards
            df.columns = [clean_column_name_for_bq(col) for col in df.columns]

            # Initialize 'show_title' column for all rows
            if 'show_title' not in df.columns:
                df['show_title'] = None

            # --- Priority 1: Extract from 'ad_name' (English terms) ---
            if 'ad_name' in df.columns:
                df['ad_name'] = df['ad_name'].astype(str)
                split_result = df['ad_name'].str.split('-', expand=True)
                df['part2'] = split_result[1].fillna('') if split_result.shape[1] > 1 else ''
                df['part3'] = split_result[2].fillna('') if split_result.shape[1] > 2 else ''
                def extract_show_title(row):
                    for part in [row.get('part2'), row.get('part3')]:
                        if pd.notna(part) and isinstance(part, str) and part in term_lookup:
                            if term_lookup[part] == 'AJ360 Branding':
                              continue
                            return term_lookup[part]
                    return None

                # Apply extraction and update 'show_title' where a value is found
                extracted_from_ad_name = df.apply(extract_show_title, axis=1)
                df.loc[extracted_from_ad_name.notna(), 'show_title'] = extracted_from_ad_name[extracted_from_ad_name.notna()]
                df.drop(columns=['part2', 'part3'], inplace=True, errors='ignore')
                print("✅ Attempted to extract 'show_title' from 'ad_name'.")
            else:
                print("⚠️ 'ad_group_name' column not found for extraction priority 1.")

            # --- Priority 2: Extract from 'ad_group_name' (English terms) ---
            if 'ad_group_name' in df.columns:
                df['ad_group_name'] = df['ad_group_name'].astype(str)
                split_result = df['ad_group_name'].str.split('_', expand=True)
                df['part2'] = split_result[1].fillna('') if split_result.shape[1] > 1 else ''
                df['part3'] = split_result[2].fillna('') if split_result.shape[1] > 2 else ''
                def extract_show_title(row):
                    for part in [row.get('part2'), row.get('part3')]:
                        if pd.notna(part) and isinstance(part, str) and part in term_lookup:
                            if term_lookup[part] == 'AJ360 Branding':
                              continue
                            return term_lookup[part]
                    return None

                # Apply extraction and update 'show_title' where a value is found
                extracted_from_ad_group = df.apply(extract_show_title, axis=1)
                df.loc[extracted_from_ad_group.notna(), 'show_title'] = extracted_from_ad_group[extracted_from_ad_group.notna()]
                df.drop(columns=['part2', 'part3'], inplace=True, errors='ignore')
                print("✅ Attempted to extract 'show_title' from 'ad_group_name'.")
            else:
                print("⚠️ 'ad_group_name' column not found for extraction priority 1.")


            # --- Priority 2: Extract from 'tweet_text' hashtags (Arabic terms), only if 'show_title' is still None ---
            if 'tweet_text' in df.columns:
                # Create a mask for rows where 'show_title' is still None (i.e., not extracted by ad_group_name)
                mask_for_hashtag_extraction = df['show_title'].isna()

                if mask_for_hashtag_extraction.any(): # Only proceed if there are rows that still need a show_title
                    df_to_process_hashtags = df.loc[mask_for_hashtag_extraction].copy()
                    df_to_process_hashtags['tweet_text'] = df_to_process_hashtags['tweet_text'].astype(str)

                    split_result_hashtags = df_to_process_hashtags['tweet_text'].apply(extract_hashtags).str.split(',', expand=True)

                    if not split_result_hashtags.empty:
                        # Helper function to safely get a column from split_result_hashtags
                        def get_col_safely(df_split, col_idx):
                            if col_idx in df_split.columns:
                                return df_split[col_idx].fillna('').str.strip()
                            # Return an empty Series of the same length if the column doesn't exist
                            return pd.Series([''] * len(df_split), index=df_split.index)

                        df_to_process_hashtags['tpart1'] = get_col_safely(split_result_hashtags, 0)
                        df_to_process_hashtags['tpart2'] = get_col_safely(split_result_hashtags, 1)
                        df_to_process_hashtags['tpart3'] = get_col_safely(split_result_hashtags, 2)
                        df_to_process_hashtags['tpart4'] = get_col_safely(split_result_hashtags, 3)
                        df_to_process_hashtags['tpart5'] = get_col_safely(split_result_hashtags, 4)
                        df_to_process_hashtags['tpart6'] = get_col_safely(split_result_hashtags, 5)
                        df_to_process_hashtags['tpart7'] = get_col_safely(split_result_hashtags, 6)
                        df_to_process_hashtags['tpart8'] = get_col_safely(split_result_hashtags, 7)
                        df_to_process_hashtags['tpart9'] = get_col_safely(split_result_hashtags, 8)
                        df_to_process_hashtags['tpart10'] = get_col_safely(split_result_hashtags, 9)

                        def extract_show_title_from_hashtag(row):
                            for part in [row.get('tpart4'), row.get('tpart5'), row.get('tpart6'), row.get('tpart7'), row.get('tpart8'), row.get('tpart9'), row.get('tpart10'), row.get('tpart3'), row.get('tpart2'), row.get('tpart1')]:
                                if pd.notna(part) and isinstance(part, str):
                                    # Apply the same cleaning to the hashtag part as to the map keys
                                    cleaned_hashtag_part = part.strip().replace(' ', '').replace('_', '')
                                    if cleaned_hashtag_part in arabic_title_to_english_title_map:
                                        return arabic_title_to_english_title_map[cleaned_hashtag_part]
                            return 'AJ360 Branding'

                        extracted_from_hashtags = df_to_process_hashtags.apply(extract_show_title_from_hashtag, axis=1)

                        # Update 'show_title' in the original DataFrame for the masked rows
                        df.loc[mask_for_hashtag_extraction & extracted_from_hashtags.notna(), 'show_title'] = extracted_from_hashtags[extracted_from_hashtags.notna()]
                        print("✅ Attempted to extract 'show_title' from 'tweet_text' hashtags for rows where it was missing.")
                    else:
                        print("⚠️ No hashtags found in 'tweet_text' for rows where 'show_title' was None.")
                else:
                    print("✅ 'show_title' already populated for all rows, skipping hashtag extraction.")
            else:
                print("⚠️ 'tweet_text' column not found for extraction priority 2.")

            # --- Align DataFrame columns with BigQuery target schema ---
            bq_schema_field_names = [field.name for field in schema]

            # Drop columns from df that are not in BQ schema
            df = df[[col for col in df.columns if col in bq_schema_field_names]]

            # Add columns to df that are in BQ schema but missing from df, as None
            for bq_col_name in bq_schema_field_names:
                if bq_col_name not in df.columns:
                    df[bq_col_name] = None

            # Reorder columns to match BigQuery schema
            df = df[bq_schema_field_names]

            # --- Type Conversion and Cleaning ---
            for field in schema:
                col = field.name
                typ = field.field_type
                if typ == 'INTEGER':
                    df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
                elif typ == 'FLOAT':
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                elif typ == 'BOOLEAN':
                    df[col] = df[col].astype('boolean')
                elif typ == 'TIMESTAMP':
                    df[col] = pd.to_datetime(df[col], errors='coerce').dt.strftime('%Y-%m-%d %H:%M:%S')
                else: # Default to string conversion for others
                    df[col] = df[col].astype(str).replace('nan', None)

            for col in df.select_dtypes(include='object').columns:
                df[col] = df[col].astype(str).str.replace('"', '').str.replace("'", '').str.replace('\n', '').str.replace('\r', '')
                df[col] = df[col].replace('nan', None)

            df.to_csv("/tmp/clean.csv", index=False, sep=',', quoting=csv.QUOTE_MINIMAL, encoding='utf-8', header=True)
            blob_clean = bucket.blob(clean_file)
            blob_clean.upload_from_filename("/tmp/clean.csv")
            print("✅ Cleaned file uploaded to GCS.")

            # --- BigQuery Load ---
            write_disposition = bigquery.WriteDisposition.WRITE_APPEND
            load_config = bigquery.LoadJobConfig(
                source_format=bigquery.SourceFormat.CSV,
                skip_leading_rows=1,
                schema=schema, # Use the determined BigQuery schema
                write_disposition=write_disposition,
                encoding='UTF-8',
                quote_character='"',
                field_delimiter=',',
                allow_quoted_newlines=True,
                allow_jagged_rows=True,
                max_bad_records=1000
            )

            destination_uri = f"gs://{bucket_name}/{clean_file}"
            load_job = bq_client.load_table_from_uri(destination_uri, table_ref, job_config=load_config)
            load_job.result()
            print(f"✅ Loaded {source_file} to BigQuery.")

        except Exception as e:
            print(f"❌ Error during file processing: {e}")
            print(traceback.format_exc()) # Print the full traceback

✅ Google Cloud authenticated successfully.
Listing CSV files in bucket: aj360_data with prefix: X_ADS/x_ads_performance_5-7Mar2026.csv
✅ Found 1 CSV files to process.
✅ Dataset aj360 exists.
Loading lookup table from BigQuery...
✅ Loaded lookup table with 699 terms.
✅ Retrieved existing BigQuery table schema for x_ads_performance.

✅ Final Schema for BigQuery operations:
  time_period: STRING
  placement: STRING
  ad_name: STRING
  tweet_text: STRING
  campaign_id: STRING
  ad_group_id: STRING
  tweet_status: STRING
  display_creative: STRING
  display_creative_status: STRING
  video_creative: STRING
  in_stream_video_status: STRING
  in_stream_video_duration: STRING
  ad_preview: STRING
  ad_type: STRING
  ad_group_name: STRING
  funding_source_name: STRING
  campaign_name: STRING
  website_creative_url: STRING
  impressions: FLOAT
  spend: FLOAT
  tweet_engagements: FLOAT
  video_views: FLOAT
  total_audience_reach: STRING
  clicks: FLOAT
  objective: STRING
  media_views: STRING
  t

In [ ]:

print(f"The variable 'extract_show_title_from_hashtag' is of type: {type(extract_show_title_from_hashtag)}")
print(f"Its value is: {extract_show_title_from_hashtag}")

print("\nLooping through the characters of 'extract_show_title_from_hashtag':")
for char in df['show_title']:
    print(char)

The variable 'extract_show_title_from_hashtag' is of type: <class 'function'>
Its value is: <function extract_show_title_from_hashtag at 0x7b39e42f7ba0>

Looping through the characters of 'extract_show_title_from_hashtag':
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
Close Your Eyes Hind
